# Implicit Euler with η-Continuation

The implicit solver with original η blows up because the hedging Hamiltonian's
gain $1/(2\eta) \approx 5 \times 10^8$ makes the sparse matrix ill-conditioned.
Policy iteration diverges because the initial guess (previous time step's θ)
is too far from the solution.

**Idea (continuation method):** At each time step, solve the implicit equation
multiple times with decreasing η scaling factors. Each sub-solve's converged θ
is used as the PI initial guess for the next (harder) sub-solve. The implicit
equation is always $\theta^{m+1} - \Delta t \cdot F_\eta[\theta^{m+1}] = \theta^m$ —
only the spatial operator $F_\eta$ changes between sub-solves; $\theta^m$ stays fixed.

This notebook tests whether η-continuation allows the implicit solver to
converge with the paper's original parameters.

In [4]:
import numpy as np
import matplotlib.pyplot as plt

from src.model import (
    build_paper_example_params, restrict_currencies, canon_pair, BP,
)
from src.policy import run_multicurrency_mm
from src.pde import (
    solve_hjb_implicit_continuation,
    solve_hjb_explicit,
    terminal_condition,
    build_pde_spec,
)

mp = restrict_currencies(build_paper_example_params(), ["USD", "EUR"])
eurusd_key = canon_pair("EUR", "USD")

print(f"Currencies: {mp.currencies}")
print(f"T = {mp.T_days} days, gamma = {mp.gamma}")
print(f"eta (original): {mp.pairs[eurusd_key].eta:.2e}")
print(f"psi:            {mp.pairs[eurusd_key].psi:.2e}")

Currencies: ['USD', 'EUR']
T = 0.05 days, gamma = 20.0
eta (original): 1.00e-09
psi:            1.00e-05


In [5]:
# Grid
Y_MAX = 150
DY = 1.0
d = len(mp.currencies)

y_grids = [np.arange(-Y_MAX, Y_MAX + DY, DY) for _ in range(d)]
idx_origin = len(y_grids[0]) // 2
y_eur = y_grids[1]

print(f"Grid: [{-Y_MAX}, {Y_MAX}] M$, dy={DY}, points/axis={len(y_grids[0])}")
print(f"Total grid points: {len(y_grids[0])**d}")

# ODE reference (original eta, for comparison)
ode_res = run_multicurrency_mm(mp)
A0 = ode_res.A0
B0 = ode_res.B0
print(f"\nODE A0[EUR,EUR] = {A0[1,1]:.6e}")
print(f"ODE B0 = {B0}")

Grid: [-150, 150] M$, dy=1.0, points/axis=301
Total grid points: 90601

ODE A0[EUR,EUR] = 9.378559e-07
ODE B0 = [0. 0.]


## 1. Step-by-step diagnostics

Manual loop that prints what happens at **each η level** of **each time step**.
This lets us see exactly where and why things break down.

In [ ]:
import warnings
import scipy.sparse.linalg

from src.pde import (
    build_pde_spec, terminal_condition, compute_gradient,
    extract_quoting_controls, extract_hedging_controls,
    assemble_implicit_system, _scale_hedging_eta,
)

N_STEPS = 20  # few steps to see what happens
ETA_SCHEDULE = (10, 5, 2, 1)
PI_TOL = 1e-10
PI_MAX_ITER = 50

T = mp.T_days
dt = T / N_STEPS

# Build specs for each eta level
spec_base = build_pde_spec(y_grids, mp)
specs = []
for s in ETA_SCHEDULE:
    if s == 1:
        specs.append(spec_base)
    else:
        specs.append(_scale_hedging_eta(spec_base, s))

kappa = mp.kappa if mp.kappa is not None else np.zeros((d, d))
theta = terminal_condition(y_grids, kappa)
grid_shape = theta.shape

print(f"dt = {dt:.4e} days, {N_STEPS} steps")
print(f"eta_schedule = {ETA_SCHEDULE}")
print(f"eta (original) = {mp.pairs[eurusd_key].eta:.2e}")
print(f"pi_max_iter = {PI_MAX_ITER}")
print()

# Header
print(f"{'Step':>4s}  {'eta_scale':>9s}  {'PI':>3s}  {'singular':>8s}  "
      f"{'NaN':>6s}  {'theta_min':>12s}  {'theta_max':>12s}  {'theta(0,0)':>12s}")
print("-" * 95)

for m in range(N_STEPS):
    theta_old = theta.copy()

    for eta_scale, spec_eta in zip(ETA_SCHEDULE, specs):
        singular = False

        for k_pi in range(PI_MAX_ITER):
            grad = compute_gradient(theta, spec_eta.dy_list)
            q_ctrl = extract_quoting_controls(theta, spec_eta)
            h_ctrl = extract_hedging_controls(grad, spec_eta.y_grids, spec_eta)

            A, source = assemble_implicit_system(q_ctrl, h_ctrl, spec_eta, dt)
            b = theta_old.ravel() + source

            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")
                theta_new = scipy.sparse.linalg.spsolve(A, b).reshape(grid_shape)
                for w in caught:
                    if "singular" in str(w.message).lower():
                        singular = True

            diff_pi = np.max(np.abs(theta_new - theta))
            scale = max(1.0, np.max(np.abs(theta_new)))
            theta = theta_new

            if diff_pi / scale < PI_TOL:
                break

        nan_count = int(np.sum(np.isnan(theta)))
        print(f"{m:4d}  {eta_scale:9d}  {k_pi+1:3d}  {str(singular):>8s}  "
              f"{nan_count:6d}  {np.nanmin(theta):12.4e}  {np.nanmax(theta):12.4e}  "
              f"{theta[idx_origin, idx_origin]:12.6e}")

        # If theta blew up (even without NaN), stop this time step
        if np.nanmax(np.abs(theta)) > 1e6:
            print(f"       >>> theta exploded at eta_scale={eta_scale}. Stopping step.")
            break

        # Stop this time step if we got NaN
        if nan_count > 0:
            break

    # Stop entirely if NaN or exploded
    if np.any(np.isnan(theta)) or np.nanmax(np.abs(theta)) > 1e6:
        print(f"\n>>> Failure at step {m}. Stopping.")
        break

print(f"\nCompleted {m+1} steps.")

In [ ]:
# PI iteration count per time step
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(pi_iters)), pi_iters, color='steelblue', width=1.0)
ax.set_xlabel('Time step')
ax.set_ylabel(f'Total PI iterations (across {len(ETA_SCHEDULE)} eta levels)')
ax.set_title(f'Policy iterations per step — schedule {ETA_SCHEDULE}')
ax.axhline(len(ETA_SCHEDULE) * 20, color='red', linestyle='--', alpha=0.5,
           label=f'Max possible ({len(ETA_SCHEDULE)} x 20 = {len(ETA_SCHEDULE)*20})')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
fig.tight_layout()
plt.show()

## 2. Compare PDE vs ODE along EUR inventory axis

The PDE solution should agree with the ODE ansatz $\hat\theta = -y^\top A_0 y - y^\top B_0 - C_0$
for moderate inventory, and diverge at large $|y|$ where the quadratic
approximation of the Hamiltonian breaks down.

In [ ]:
# Slice along EUR axis (y_USD = 0)
slice_pde = theta_cont[idx_origin, :]

# ODE ansatz: theta_hat = -y^T A0 y - y^T B0 - C0
# Along y_USD=0: theta_hat = -A0[1,1] * y_EUR^2 - B0[1] * y_EUR - C0
# C0 is not directly available; shift to match at origin
theta_ode_raw = -A0[1, 1] * y_eur**2 - B0[1] * y_eur
C0_shift = slice_pde[idx_origin] - theta_ode_raw[idx_origin]
theta_ode = theta_ode_raw + C0_shift

roi = np.abs(y_eur) <= 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(y_eur[roi], slice_pde[roi], 'b-', linewidth=2, label='PDE (continuation)')
ax1.plot(y_eur[roi], theta_ode[roi], 'r--', linewidth=2, label='ODE ansatz (shifted)')
ax1.set_xlabel('$y_{\\mathrm{EUR}}$ (M$)', fontsize=12)
ax1.set_ylabel('$\\theta(0, y)$', fontsize=12)
ax1.set_title('Value function: PDE vs ODE', fontsize=13)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

diff = slice_pde[roi] - theta_ode[roi]
ax2.plot(y_eur[roi], diff, 'k-', linewidth=1.5)
ax2.set_xlabel('$y_{\\mathrm{EUR}}$ (M$)', fontsize=12)
ax2.set_ylabel('$\\theta_{\\mathrm{PDE}} - \\hat\\theta_{\\mathrm{ODE}}$', fontsize=12)
ax2.set_title('Difference (PDE - ODE)', fontsize=13)
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

print(f"theta_PDE at origin:  {slice_pde[idx_origin]:.6e}")
print(f"theta_ODE at origin:  {theta_ode[idx_origin]:.6e}  (shifted to match)")
print(f"Max |diff| in [-100, 100]: {np.max(np.abs(diff)):.6e}")

In [ ]:
# 2D heatmap
roi_idx = np.abs(y_grids[0]) <= 100
roi_2d = np.ix_(roi_idx, roi_idx)
y_roi = y_grids[0][roi_idx]
extent = [y_roi[0], y_roi[-1], y_roi[0], y_roi[-1]]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

im1 = ax1.imshow(theta_cont[roi_2d].T, origin='lower', extent=extent,
                  aspect='equal', cmap='viridis')
ax1.set_xlabel('$y_{\\mathrm{USD}}$ (M$)')
ax1.set_ylabel('$y_{\\mathrm{EUR}}$ (M$)')
ax1.set_title('$\\theta(0, y)$ — PDE continuation', fontsize=13)
plt.colorbar(im1, ax=ax1, shrink=0.8)

# ODE ansatz on full 2D grid
Y_USD, Y_EUR = np.meshgrid(y_grids[0], y_grids[1], indexing='ij')
theta_ode_2d = -(A0[0,0]*Y_USD**2 + 2*A0[0,1]*Y_USD*Y_EUR + A0[1,1]*Y_EUR**2) \
               - (B0[0]*Y_USD + B0[1]*Y_EUR) + C0_shift

diff_2d = theta_cont[roi_2d] - theta_ode_2d[roi_2d]
vabs = max(abs(diff_2d.min()), abs(diff_2d.max()))
im2 = ax2.imshow(diff_2d.T, origin='lower', extent=extent,
                  aspect='equal', cmap='RdBu_r',
                  vmin=-vabs, vmax=vabs)
ax2.set_xlabel('$y_{\\mathrm{USD}}$ (M$)')
ax2.set_ylabel('$y_{\\mathrm{EUR}}$ (M$)')
ax2.set_title('$\\theta_{\\mathrm{PDE}} - \\hat\\theta_{\\mathrm{ODE}}$', fontsize=13)
plt.colorbar(im2, ax=ax2, shrink=0.8)

fig.tight_layout()
plt.show()

## 3. Sanity check: does η×1000 alone still match the explicit solver?

Run the continuation solver with schedule `(1000,)` (no continuation, just
η×1000) and compare against the explicit solver with η×1000.
This verifies the continuation infrastructure doesn't break anything.

In [ ]:
from src.model import PairParams

# Build scaled-eta model for explicit solver comparison
mp_scaled = restrict_currencies(build_paper_example_params(), ["USD", "EUR"])
new_pairs = {}
for key, pp in mp_scaled.pairs.items():
    new_pairs[key] = PairParams(
        pair=pp.pair, sizes_musd=pp.sizes_musd,
        lambdas_per_day=pp.lambdas_per_day, tiers=pp.tiers,
        psi=pp.psi, eta=pp.eta * 1000,
    )
mp_scaled.pairs = new_pairs

# Explicit solver (eta x 1000, 1000 steps)
print("Running explicit solver (eta x 1000, 1000 steps)...")
res_exp = solve_hjb_explicit(y_grids, mp_scaled, n_steps=1000)
theta_exp = res_exp['theta_0']

# Continuation solver with schedule=(1000,) — should match
# Use same step count for fair comparison
print("Running continuation solver with schedule=(1000,), 1000 steps...")
res_cont_1000 = solve_hjb_implicit_continuation(
    y_grids, mp, n_steps=1000,
    eta_schedule=(1000,),
    pi_tol=1e-10, pi_max_iter=20,
)
theta_cont_1000 = res_cont_1000['theta_0']

slice_exp = theta_exp[idx_origin, :]
slice_cont = theta_cont_1000[idx_origin, :]

print(f"\nExplicit theta_0(0,0):     {theta_exp[idx_origin, idx_origin]:.6e}")
print(f"Continuation theta_0(0,0): {theta_cont_1000[idx_origin, idx_origin]:.6e}")
print(f"Max |diff| in [-100,100]:  {np.max(np.abs(slice_exp[roi] - slice_cont[roi])):.6e}")

## 4. Try different schedules

If the main run above failed or hit max PI, try finer schedules.
If it worked easily, try coarser schedules to find the minimal one.

In [ ]:
# --- Uncomment and run whichever schedule you want to try ---

# Finer schedule (if (1000, 100, 10, 1) blows up)
# ETA_FINE = (1000, 300, 100, 30, 10, 3, 1)

# Coarser schedule (if (1000, 100, 10, 1) works easily)
# ETA_COARSE = (100, 1)

# Minimal: just one warm-start level
# ETA_MINIMAL = (10, 1)

schedule_to_try = (1000, 100, 10, 1)  # <-- change this

print(f"Testing schedule: {schedule_to_try}")
res_test = solve_hjb_implicit_continuation(
    y_grids, mp,
    n_steps=N_STEPS,
    eta_schedule=schedule_to_try,
    pi_tol=1e-10,
    pi_max_iter=20,
)

theta_test = res_test['theta_0']
pi_test = res_test['pi_iters']

has_nan = np.any(np.isnan(theta_test))
print(f"\nHas NaN: {has_nan}")
print(f"theta_0 at origin: {theta_test[idx_origin, idx_origin]:.6e}")
print(f"PI iters: min={min(pi_test)}, max={max(pi_test)}, mean={np.mean(pi_test):.1f}")

if not has_nan:
    slice_test = theta_test[idx_origin, :]
    diff_test = slice_test[roi] - theta_ode[roi]
    print(f"Max |PDE - ODE| in [-100, 100]: {np.max(np.abs(diff_test)):.6e}")